<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 · My Fund vs the Crowd (DRAFT)

Every Studio 5 fund on the leaderboard is one vote. Average all of the votes and you get the **class crowd fund**.
Tonight you work with a partner on two questions:

> **1. Did my fund beat the crowd, and why?**
> **2. What if we bet big on a few great companies we understand?**

| | Part | What you do | About |
|---|---|---|---|
| 1 | **Load our funds** | each partner pastes their record link; Run all | 10 min |
| 2 | **Me vs the crowd** | read where your fund ranks, the risk-return map, and what drove it | 35 min |
| 3 | **Our concentrated fund 🎯** | pick 3 to 6 companies you understand, write a thesis for each, and send the fund to the board | 50 min |
| 4 | **The call** | read six market signals and say which fund you think wins by Dec 3 | 15 min |

Each partner runs this notebook on their own laptop, with their own fund. You build the 🎯 fund together, and one of
you sends it.

**This notebook is also your final project.** After the Dec 3 close you Run all again, and the notebook prints every
number your 2-page report needs, plus a one-page fact sheet for your resume.

## Import modules (download Pyomo, ipopt and yfinance)

Same setup as Studio 5. Run this first, every time. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import json
import time
import requests
from pyomo.environ import *
from IPython.display import display
import matplotlib.dates as mdates

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · Load our funds

Your Studio 5 receipt printed a line called **your private record**: a long link that ends in `.json`. Paste yours
below. If your partner shares theirs with you, paste it as `partner_record_link` and you will see both funds side by
side. (Keep record links between you and your partner.)

**No Studio 5 fund?** Leave `record_link` blank and pick a Studio 5 list with `my_theme`. The notebook splits the money
evenly across it, so you can still do every part tonight.

In [ ]:
record_link = ''                 # paste the "your private record" link from your Studio 5 receipt
partner_record_link = ''         # optional: your partner's record link
concentrated_record_link = ''    # after Studio 6: the 🎯 fund's record link (from the receipt of whoever sent it)

# only used when record_link is blank: 'starter' or a theme ('connecticut', 'ai_chips', 'oil_and_gas',
# 'banks_and_payments', 'dividends', 'healthcare', 'food_and_restaurants', 'travel', 'defense_and_aerospace',
# 'utilities_and_power', 'shopping', 'sports_and_entertainment')
my_theme = 'starter'

Run the next cell as it is. It reads your record, your partner's record and the class crowd fund.

In [ ]:
# the Studio 5 starter list and the twelve themes (every company was in the S&P 500 on Dec 31, 2025)
starter = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'GS', 'V', 'JNJ', 'LLY', 'UNH', 'XOM', 'CVX', 'KO', 'WMT',
           'COST', 'AMZN', 'HD', 'CAT', 'LMT']
themes = {
    'connecticut': ['TRV', 'HIG', 'CI', 'SWK', 'RTX', 'OTIS', 'CARR', 'URI', 'SYF', 'HUBB', 'ES', 'IT', 'FDS', 'CHTR', 'XOM', 'PG', 'MSFT', 'UNH'],
    'ai_chips': ['NVDA', 'AMD', 'AVGO', 'MU', 'LRCX', 'AMAT', 'KLAC', 'INTC', 'QCOM', 'TXN', 'ADI', 'ANET', 'MSFT', 'GOOGL', 'META', 'ORCL', 'LLY', 'XOM'],
    'oil_and_gas': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'MPC', 'VLO', 'KMI', 'WMB', 'OKE', 'HAL', 'DVN', 'FANG', 'MSFT', 'JNJ'],
    'banks_and_payments': ['JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'SCHW', 'V', 'MA', 'AXP', 'PYPL', 'COF', 'BLK', 'USB', 'AAPL', 'UNH', 'CVX'],
    'dividends': ['KO', 'PEP', 'PG', 'JNJ', 'MRK', 'ABBV', 'VZ', 'T', 'DUK', 'SO', 'XOM', 'CVX', 'MMM', 'IBM', 'MO', 'PM', 'CSCO', 'O'],
    'healthcare': ['LLY', 'JNJ', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'ISRG', 'AMGN', 'GILD', 'CVS', 'MDT', 'BSX', 'SYK', 'AAPL', 'XOM'],
    'food_and_restaurants': ['MCD', 'SBUX', 'CMG', 'YUM', 'DPZ', 'KO', 'PEP', 'MDLZ', 'HSY', 'GIS', 'KHC', 'KDP', 'DRI', 'TSN', 'COST', 'AAPL', 'JNJ', 'XOM'],
    'travel': ['BKNG', 'MAR', 'HLT', 'ABNB', 'EXPE', 'DAL', 'UAL', 'LUV', 'RCL', 'CCL', 'NCLH', 'LVS', 'WYNN', 'MGM', 'UBER', 'AAPL', 'JNJ', 'XOM'],
    'defense_and_aerospace': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'LHX', 'HII', 'TXT', 'GE', 'HON', 'TDG', 'HWM', 'PLTR', 'LDOS', 'XOM', 'JNJ', 'MSFT'],
    'utilities_and_power': ['NEE', 'DUK', 'SO', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'CEG', 'VST', 'ETR', 'FSLR', 'AES', 'XOM', 'JNJ', 'MSFT'],
    'shopping': ['AMZN', 'WMT', 'COST', 'TGT', 'HD', 'LOW', 'TJX', 'NKE', 'ROST', 'ULTA', 'DG', 'DLTR', 'BBY', 'EBAY', 'AAPL', 'CVS', 'XOM'],
    'sports_and_entertainment': ['DIS', 'NFLX', 'LYV', 'NKE', 'TTWO', 'CMCSA', 'CHTR', 'MAR', 'RCL', 'LVS', 'FOXA', 'HAS', 'GOOGL', 'META', 'AAPL', 'JNJ', 'XOM'],
}

# every S&P 500 company's sector (technology is called 'Technology', as in Studio 5)
members = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv')
sp500_sectors = {}
for i in range(len(members)):
    sector = members['sector'].iloc[i]
    if sector == 'Information Technology':
        sector = 'Technology'
    sp500_sectors[members['ticker'].iloc[i]] = sector

CROWD_LINK = 'https://storage.googleapis.com/opim5641-leaderboard-f26/crowd.json'


def read_json(link):
    # a record or the crowd: a web link, or a file in this folder
    if link.strip().startswith('http'):
        return requests.get(link.strip(), timeout=30).json()
    return json.load(open(link.strip()))


def latest_fund(record):
    # the latest diversified (Studio 5) mix in a record: {ticker: share}, plus the entry it came from
    found = None
    for entry in record['entries']:
        if entry.get('fund_kind', 'diversified') != 'concentrated' and len(entry['portfolios'].get('with_rules', {})) > 0:
            found = entry
    if found is None:
        return None, None
    return found['portfolios']['with_rules'], found


my_record = None
my_entry = None
if record_link.strip() != '':
    my_record = read_json(record_link)
    my_mix, my_entry = latest_fund(my_record)
if my_entry is None:
    if my_theme in themes:
        picked = themes[my_theme]
    else:
        my_theme = 'starter'
        picked = starter
    my_mix = {}
    for t in picked:
        my_mix[t] = 1 / len(picked)
    my_name = 'my fund (%s, even split)' % my_theme
    print("No record loaded, so your fund is an even split of the", my_theme, "list.")
else:
    my_name = my_record.get('nickname', 'my fund')
    my_theme = my_entry.get('stock_theme') or 'custom'
    print("Loaded your fund:", my_name, "|", len(my_mix), "stocks")

partner_record = None
partner_mix = None
partner_name = None
if partner_record_link.strip() != '':
    partner_record = read_json(partner_record_link)
    partner_mix, partner_entry = latest_fund(partner_record)
    partner_name = partner_record.get('nickname', 'my partner')
    print("Loaded your partner's fund:", partner_name, "|", len(partner_mix), "stocks")

crowd_file = read_json(CROWD_LINK)
if crowd_file.get('crowd') is None:
    raise RuntimeError("The class crowd fund is not ready yet: it starts once student funds are on the leaderboard.")
crowd_mix = crowd_file['crowd']['weights']
class_funds = crowd_file['funds']
print("The class crowd fund: the average of", crowd_file['crowd']['members'], "student funds,",
      len(crowd_mix), "stocks in all", "(frozen)" if crowd_file['frozen'] else "(still open until the end of Nov 18)")

### Daily prices

Daily adjusted closing prices from October 2025 to the latest close, for every stock anyone in the class holds, plus the
S&P 500 (SPY). If Yahoo blocks Colab, the cell uses the class copy.

In [ ]:
import yfinance as yf

saved_prices = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)


def get_prices(tickers):
    # daily closes for these tickers (+ SPY) from Oct 2025; the class copy fills in anything Yahoo will not give us
    wanted = ['SPY']
    for t in tickers:
        if t not in wanted:
            wanted.append(t)
    table = None
    try:
        data = yf.download(wanted, start='2025-10-01', interval='1d', auto_adjust=True, progress=False)
        table = data['Close']
    except Exception as problem:
        table = None
    if table is None or 'SPY' not in table.columns or table.notna().any().mean() < 0.8:
        print("Yahoo did not give us enough stocks, so we are using the class copy instead.")
        table = pd.DataFrame(index=saved_prices.loc['2025-10-01':].index)
    missing = []
    for t in wanted:
        if t not in table.columns or table[t].notna().sum() == 0:
            if t in saved_prices.columns:
                table[t] = saved_prices[t].reindex(table.index)
            else:
                missing.append(t)
    if len(missing) > 0:
        print("No prices for:", missing, "(held as cash)")
    table = table.ffill()
    # today's row is not a close until the market has closed, so we stop at yesterday
    table = table[table.index < pd.Timestamp.today().normalize()]
    return table


everyone = list(my_mix) + list(crowd_mix)
if partner_mix is not None:
    everyone = everyone + list(partner_mix)
for fund in class_funds:
    everyone = everyone + list(fund)
prices = get_prices(everyone)
days = prices.loc['2026-01-01':].index          # every trading day of 2026 so far
print(prices.shape[0], "trading days for", prices.shape[1] - 1, "stocks + SPY, through", prices.index[-1].date())

# Part 2 · Me vs the crowd

**The yardstick.** To compare every fund fairly, we hold each fund's mix (its shares right now) through all of 2026
so far, starting with $10,000 on Jan 2. It is a *what-if*: the mixes were chosen in November, so they had some of 2026
in view. The live race (later in this part) is the honest test, but it is only a few weeks long.

**The risk-return map** puts every fund in the class on one picture:

* **across:** risk, Ms. Womack's measure (how much the fund's value bounces from day to day, per year, in %);
* **up:** return, what $10,000 grew by in 2026 so far.

Two facts about the crowd fund are always true, and the map shows both:

1. Its **return is exactly the class average.** (Owning 1/N of every fund means you get 1/N of every fund's gain.)
2. Its **risk is never more than the class's average risk**, and usually a lot less. Different funds have bad days on
   different days, and when you mix them some of the ups and downs cancel out. That is diversification.

The gap between the two is the **diversification gap: lower risk for the same return.**

In [ ]:
start_day = days[0]
end_day = days[-1]
daily = prices.loc[start_day:end_day].pct_change().iloc[1:]
cov_2026 = daily.cov() * 252


def grow(mix):
    # $10,000 in this mix on Jan 2, held (buy and hold) to the latest close; a stock with no price is held as cash
    path = pd.Series(0.0, index=days)
    for t in mix:
        if t in prices.columns and prices.loc[start_day, t] == prices.loc[start_day, t]:
            path += 10000 * mix[t] * prices.loc[days, t] / prices.loc[start_day, t]
        else:
            path += 10000 * mix[t]
    return path


def risk_of(mix):
    # Ms. Womack's risk of this mix on 2026's daily returns, per year, in %
    names = []
    for t in mix:
        if t in cov_2026.columns:
            names.append(t)
    total = 0
    for i in names:
        for j in names:
            total += mix[i] * cov_2026.loc[i, j] * mix[j]
    return 100 * np.sqrt(total)


def worst_drop(path):
    # the biggest fall from a high point, in %
    return float(100 * (1 - path / path.cummax()).max())


def card(mix, path):
    return {'2026 return (%)': round(100 * (path.iloc[-1] / 10000 - 1), 1), 'risk (%)': round(risk_of(mix), 1),
            'worst drop (%)': round(worst_drop(path), 1), '$10,000 became': round(path.iloc[-1], 0), 'stocks': len(mix)}


def same_mix(a, b):
    # True when two mixes are the same fund (to find yours among the class funds)
    for t in set(list(a) + list(b)):
        if abs(a.get(t, 0) - b.get(t, 0)) > 0.0005:
            return False
    return True


class_rows = []
mine_found = False
for fund in class_funds:
    row = card(fund, grow(fund))
    row['mine'] = same_mix(fund, my_mix)
    if row['mine']:
        mine_found = True
    class_rows.append(row)
class_table = pd.DataFrame(class_rows)

my_path = grow(my_mix)
crowd_path = grow(crowd_mix)
spy_path = 10000 * prices.loc[days, 'SPY'] / prices.loc[start_day, 'SPY']
mine = card(my_mix, my_path)
crowd = card(crowd_mix, crowd_path)
spy = card({'SPY': 1.0}, spy_path)
average_return = class_table['2026 return (%)'].mean()
average_risk = class_table['risk (%)'].mean()

others = class_table[class_table['mine'] == False]
my_rank = 1 + int((others['2026 return (%)'] > mine['2026 return (%)']).sum())
how_many = len(others) + 1
safer_than = int((class_table['risk (%)'] > crowd['risk (%)']).sum())

lines = {my_name: mine, 'class crowd fund': crowd, 'S&P 500 (SPY)': spy,
         'class average (%d funds)' % len(class_table): {'2026 return (%)': round(average_return, 1), 'risk (%)': round(average_risk, 1)}}
if partner_mix is not None:
    partner_path = grow(partner_mix)
    lines[partner_name] = card(partner_mix, partner_path)
scorecard = pd.DataFrame(lines).T
print("2026 so far (what-if: each fund's mix held from Jan 2), through", end_day.date())
print("Your fund ranks #%d of %d on return." % (my_rank, how_many))
print("Check: the crowd's return %.1f%% = the class average %.1f%%. Its risk %.1f%% vs the class average risk %.1f%%." % (
    crowd['2026 return (%)'], average_return, crowd['risk (%)'], average_risk))
scorecard

In [ ]:
def draw_map(extra=None, panel=None):
    # every fund in the class as a dot: risk across, 2026 return up
    if panel is None:
        fig, panel = subplots(figsize=(11, 7))
    panel.scatter(class_table['risk (%)'], class_table['2026 return (%)'], s=60, color='lightgray', edgecolor='gray',
                  label='a fund in the class', zorder=2)
    panel.axvline(average_risk, color='gray', linestyle='--', linewidth=1)
    panel.axhline(average_return, color='gray', linestyle='--', linewidth=1)
    panel.annotate('', xy=(crowd['risk (%)'], crowd['2026 return (%)']), xytext=(average_risk, average_return),
                   arrowprops={'arrowstyle': '-|>', 'color': 'tab:purple', 'linewidth': 2, 'shrinkB': 14})
    panel.text((average_risk + crowd['risk (%)']) / 2, average_return, 'the diversification gap\n',
               color='tab:purple', ha='center', va='bottom', fontsize=10)
    panel.scatter([crowd['risk (%)']], [crowd['2026 return (%)']], s=260, color='tab:purple', marker='h',
                  label='class crowd fund', zorder=4)
    panel.scatter([spy['risk (%)']], [spy['2026 return (%)']], s=150, color='black', marker='s', label='S&P 500', zorder=4)
    if partner_mix is not None:
        panel.scatter([lines[partner_name]['risk (%)']], [lines[partner_name]['2026 return (%)']], s=200,
                      color='tab:orange', marker='D', label=partner_name, zorder=5)
    panel.scatter([mine['risk (%)']], [mine['2026 return (%)']], s=380, color='tab:green', marker='*', label=my_name, zorder=6)
    if extra is not None:
        panel.scatter([extra['risk (%)']], [extra['2026 return (%)']], s=300, color='tab:red', marker='P',
                      label=extra['label'], zorder=6)
    panel.set_xlabel('risk: how much the value bounces, per year (%)')
    panel.set_ylabel('2026 return so far (%)')
    panel.set_title('The class on the risk-return map (dashed lines: the class averages)')
    panel.legend(fontsize=9, loc='best')
    return panel


draw_map()
show()
print("The crowd is less risky than %d of %d funds, with the same return as the average fund (%.1f%%)." % (
    safer_than, len(class_table), average_return))

**How to read it.** Up is good and left is calm. A fund up and to the left of the purple crowd did better than the
crowd on both counts. Most funds sit to the right of the crowd: they took more risk than the crowd for a return near
the class average.

### What drove my fund?

Three cuts at the same question:

* **Helpers and hurters:** each stock's dollars = its share × $10,000 × how much it moved.
* **My bets vs the crowd:** where your shares differ from the crowd's, by sector. The dollars add up exactly to the gap
  between your fund and the crowd.
* **Overlap:** how many of the crowd's ten biggest holdings you hold too.

In [ ]:
moved = {}
for t in set(list(my_mix) + list(crowd_mix)):
    if t in prices.columns and prices.loc[start_day, t] == prices.loc[start_day, t]:
        moved[t] = prices.loc[end_day, t] / prices.loc[start_day, t] - 1
    else:
        moved[t] = 0.0

dollars = {}
for t in my_mix:
    dollars[t] = 10000 * my_mix[t] * moved[t]
dollars = pd.Series(dollars).sort_values(ascending=False)
helpers = dollars.head(3)
hurters = dollars.tail(3).sort_values()

bets = {}
for t in set(list(my_mix) + list(crowd_mix)):
    sector = sp500_sectors.get(t, 'Other')
    if sector not in bets:
        bets[sector] = {'my share (%)': 0.0, 'crowd share (%)': 0.0, 'effect on my gap ($)': 0.0}
    bets[sector]['my share (%)'] += 100 * my_mix.get(t, 0)
    bets[sector]['crowd share (%)'] += 100 * crowd_mix.get(t, 0)
    bets[sector]['effect on my gap ($)'] += 10000 * (my_mix.get(t, 0) - crowd_mix.get(t, 0)) * moved[t]
bets = pd.DataFrame(bets).T
bets['my bet (points)'] = bets['my share (%)'] - bets['crowd share (%)']
bets = bets.sort_values('effect on my gap ($)', ascending=False).round(1)

crowd_top = pd.Series(crowd_mix).sort_values(ascending=False).head(10)
shared = []
for t in crowd_top.index:
    if t in my_mix:
        shared.append(t)

gap = my_path.iloc[-1] - crowd_path.iloc[-1]
print("My fund vs the crowd in 2026 so far: %s$%.0f" % ('+' if gap >= 0 else '-', abs(gap)))
print()
for t in helpers.index:
    print("  helped:  %-6s %s  %+.0f dollars" % (t, sp500_sectors.get(t, 'Other'), helpers[t]))
for t in hurters.index:
    if hurters[t] < 0:
        print("  hurt:    %-6s %s  %+.0f dollars" % (t, sp500_sectors.get(t, 'Other'), hurters[t]))
    else:
        print("  helped least: %-6s %s  %+.0f dollars" % (t, sp500_sectors.get(t, 'Other'), hurters[t]))
print()
print("I hold %d of the crowd's 10 biggest holdings: %s" % (len(shared), ", ".join(shared) if len(shared) > 0 else "none"))
print("The sector bets below add up to the gap: $%.0f" % bets['effect on my gap ($)'].sum())
bets[['my share (%)', 'crowd share (%)', 'my bet (points)', 'effect on my gap ($)']]

### The live race

The honest test: your real fund on the leaderboard since it started, against the class crowd fund (held since the
Studio 5 makeup, Nov 5) and the S&P 500. A few weeks is a short race, so most of any gap is luck.

In [ ]:
def live_line(record):
    # a fund's value on the leaderboard since it started (None before two closes)
    if record is None or len(record['values']['dates']) < 2:
        return None
    return pd.Series(record['values']['with_rules'], index=pd.to_datetime(record['values']['dates']))


my_live = live_line(my_record)
partner_live = live_line(partner_record)
crowd_live = None
if crowd_file['crowd']['path'] is not None and len(crowd_file['crowd']['path']['dates']) >= 2:
    crowd_live = pd.Series(crowd_file['crowd']['path']['values'], index=pd.to_datetime(crowd_file['crowd']['path']['dates']))

if my_live is None:
    print("Paste your record link at the top to see your fund's live race.")
else:
    spy_live = 10000 * prices.loc[my_live.index[0]:, 'SPY'] / prices.loc[my_live.index[0], 'SPY']
    fig, panel = subplots(figsize=(11, 5))
    panel.plot(my_live.index, my_live.values, color='tab:green', linewidth=2.5, marker='o', label=my_name)
    if partner_live is not None:
        panel.plot(partner_live.index, partner_live.values, color='tab:orange', linewidth=2, marker='o', label=partner_name)
    if crowd_live is not None:
        panel.plot(crowd_live.index, crowd_live.values, color='tab:purple', linewidth=2.5, linestyle='--', label='class crowd fund')
    panel.plot(spy_live.index, spy_live.values, color='black', linewidth=2, label='S&P 500')
    panel.axhline(10000, color='lightgray', linestyle=':')
    panel.set_title('The live race on the class leaderboard ($10,000 at the start)')
    panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    panel.legend(fontsize=9)
    show()
    print("Since %s: my fund $%.0f, S&P 500 $%.0f" % (my_live.index[0].date(), my_live.iloc[-1], spy_live.iloc[-1]))
    if crowd_live is not None:
        print("The crowd since %s: $%.0f" % (crowd_live.index[0].date(), crowd_live.iloc[-1]))

### Talk with your partner (10 minutes)

1. Whose fund beat the crowd in 2026 so far? In the live race?
2. How many stocks do your two funds share? Which bet came from each of you?
3. Where are your two dots on the map, left or right of the crowd? Was the extra risk worth it?

# Part 3 · Our concentrated fund 🎯

The podcast's advice: **buy great companies you understand, and hold a few of them with conviction.** Your Studio 5
fund spread its bets. Now the two of you bet big.

**The rules for a 🎯 fund:**

| Rule | Setting |
|---|---|
| Companies | 3 to 6, any company in the S&P 500 |
| A thesis for each | one line: *"We understand it because ___, and we think it grows because ___."* |
| Every pick is held | at least 5% in each (you picked it, so you own it) |
| The cap | at most **40%** in any one company |
| The risk limit | `'off'` (go for the most return) or `'2x'` (at most twice the risk of an even split) |

Ms. Womack's model picks the shares inside those rules: the highest average daily return over the last year (252
trading days), for the risk you allow.

The example below runs as it is. **Replace it with your own picks and theses** before you send it.

In [ ]:
concentrated = {
    'name': 'Our Conviction Fund',     # the fund's name on the board (2 to 30 characters; the board adds 🎯)
    'picks': {
        'COST': 'We understand it because ___, and we think it grows because ___',
        'AAPL': 'We understand it because ___, and we think it grows because ___',
        'JPM': 'We understand it because ___, and we think it grows because ___',
    },
    'risk_limit': '2x',                # 'off' or '2x'
}

focus_rules = {
    'min_companies': 3,
    'max_companies': 6,
    'min_share': 0.05,
    'max_share': 0.40,
}

In [ ]:
def solve_concentrated(picks, risk_setting):
    # Ms. Womack's model on the picks: the most return, every pick held (5% to 40%), and the risk limit you chose
    names = list(picks)
    window = prices[names].pct_change().iloc[1:].tail(252)
    df_return = window.mean()
    df_cov = window.cov()

    even_risk = 0
    for i in names:
        for j in names:
            even_risk += df_cov.loc[i, j] / len(names) ** 2

    model = ConcreteModel()
    model.x = Var(names, domain=NonNegativeReals, bounds=(focus_rules['min_share'], focus_rules['max_share']))
    for s in names:
        model.x[s] = 1 / len(names)

    obj_expr = 0
    for s in names:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()
    budget = 0
    for s in names:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    risk_expr = 0
    for i in names:
        for j in names:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    if risk_setting == '2x':
        model.constraints.add(risk_expr <= 2 * even_risk)

    SolverFactory('ipopt', executable=ipopt_path).solve(model)
    weights = {}
    for s in names:
        weights[s] = round(model.x[s](), 6)
    return weights, 100 * np.sqrt(252 * value(risk_expr)), 100 * np.sqrt(252 * even_risk)


problems = []
picks = {}
for t in concentrated['picks']:
    picks[t.strip().upper()] = concentrated['picks'][t].strip()
if len(picks) < focus_rules['min_companies'] or len(picks) > focus_rules['max_companies']:
    problems.append("pick 3 to 6 companies")
for t in picks:
    if t not in prices.columns:
        prices = get_prices(list(prices.columns) + [t])
    if t not in prices.columns:
        problems.append("no prices for %s: check the ticker" % t)
if concentrated['risk_limit'] not in ['off', '2x']:
    problems.append("risk_limit should be 'off' or '2x'")

if len(problems) > 0:
    print("Fix this first:", "; ".join(problems))
else:
    focus_mix, focus_risk_window, even_risk_window = solve_concentrated(picks, concentrated['risk_limit'])
    # quality control: every rule, checked on the answer
    total = 0
    for t in focus_mix:
        total += focus_mix[t]
        if focus_mix[t] < focus_rules['min_share'] - 1e-4 or focus_mix[t] > focus_rules['max_share'] + 1e-4:
            problems.append("%s has %.1f%%" % (t, 100 * focus_mix[t]))
    if abs(total - 1) > 1e-4:
        problems.append("the shares add up to %.4f" % total)
    if concentrated['risk_limit'] == '2x' and focus_risk_window > 2 * even_risk_window + 0.01:
        problems.append("the risk is over the limit")
    print("QC:", "PASS" if len(problems) == 0 else "FAIL: " + "; ".join(problems))
    print("risk over the last year: %.1f%% a year (an even split of your picks: %.1f%%)" % (focus_risk_window, even_risk_window))
    shares = pd.DataFrame({'share (%)': pd.Series(focus_mix) * 100, 'thesis': pd.Series(picks)})
    shares['sector'] = pd.Series(sp500_sectors).reindex(shares.index)
    display(shares.sort_values('share (%)', ascending=False).round(1))

### Concentrated vs diversified

The same yardstick as Part 2: each mix held through 2026 so far from Jan 2.

In [ ]:
focus_path = grow(focus_mix)
focus = card(focus_mix, focus_path)
focus['label'] = 'concentrated: ' + concentrated['name']    # (chart fonts have no 🎯)

compare = {focus['label']: focus, my_name: mine, 'class crowd fund': crowd, 'S&P 500 (SPY)': spy}
if partner_mix is not None:
    compare[partner_name] = lines[partner_name]
compare_table = pd.DataFrame(compare).T.drop(columns=['label'], errors='ignore')
compare_table['stocks'] = compare_table['stocks'].astype(int)

draw_map(focus)
show()
print("The 🎯 fund vs your fund: %s$%.0f in 2026 so far, with %.1f points %s risk." % (
    '+' if focus_path.iloc[-1] >= my_path.iloc[-1] else '-', abs(focus_path.iloc[-1] - my_path.iloc[-1]),
    abs(focus['risk (%)'] - mine['risk (%)']), 'more' if focus['risk (%)'] > mine['risk (%)'] else 'less'))
compare_table

Where did the 🎯 fund land on the map? Concentrated funds usually sit far to the right: more risk. Whether the extra
risk pays off is what the next two weeks will show.

# Part 4 · The call

Time to switch hats. You are on the **investment committee**, and the question is about the next two weeks: **is
the market in a bull, a bear, or going sideways?** Nobody knows for sure. Investors look at a few simple signals, and
so will you. Every signal uses only prices through the latest close.

| Signal | 🐂 bull if... | 🐻 bear if... |
|---|---|---|
| **Trend** | the S&P 500 is above its 200-day average | it is below it |
| **Crossover** | the 50-day average is above the 200-day average (a "golden cross") | it is below (a "death cross") |
| **Momentum** | the S&P 500 is up more than 3% over the last 3 months (63 trading days) | down more than 3% |
| **Calm or stormy** | daily ups and downs are calmer than usual for the past year | stormier than usual (25% or more above normal) |
| **Distance from the high** | less than 5% below its high of the past year | more than 10% below it |
| **Breadth (your stocks)** | more than 60% of your stocks are above their own 200-day average | fewer than 40% are |

Anything in between counts as ➖ sideways.

In [ ]:
# the S&P 500 since Oct 2024: the class copy, then the fresh prices on top
spy_line = pd.concat([saved_prices['SPY'], prices['SPY']])
spy_line = spy_line[~spy_line.index.duplicated(keep='last')].sort_index().dropna()
signals_from = spy_line.index[-1] - pd.Timedelta(days=400)
average_50 = spy_line.rolling(50).mean()
average_200 = spy_line.rolling(200).mean()
stormy = spy_line.pct_change().rolling(21).std() * np.sqrt(252) * 100
normal = stormy.tail(252).median()
high_252 = spy_line.rolling(252, min_periods=120).max()
below_high = 100 * (1 - spy_line / high_252)

breadth_rows = {}
for t in my_mix:
    if t in prices.columns:
        breadth_rows[t] = 100 * (prices[t].iloc[-1] / prices[t].tail(200).mean() - 1)
breadth = pd.Series(breadth_rows).sort_values()
share_above = 100 * (breadth > 0).mean()
momentum_3m = 100 * (spy_line.iloc[-1] / spy_line.iloc[-64] - 1)


def read(bull, bear):
    # one signal's reading
    if bull:
        return '🐂 bull'
    if bear:
        return '🐻 bear'
    return '➖ sideways'


signals = pd.DataFrame([
    {'signal': 'Trend', 'reading': 'S&P 500 %.0f vs 200-day average %.0f' % (spy_line.iloc[-1], average_200.iloc[-1]),
     'call': read(spy_line.iloc[-1] > average_200.iloc[-1], spy_line.iloc[-1] < average_200.iloc[-1])},
    {'signal': 'Crossover', 'reading': '50-day %.0f vs 200-day %.0f' % (average_50.iloc[-1], average_200.iloc[-1]),
     'call': read(average_50.iloc[-1] > average_200.iloc[-1], average_50.iloc[-1] < average_200.iloc[-1])},
    {'signal': 'Momentum', 'reading': '%+.1f%% over the last 63 trading days' % momentum_3m,
     'call': read(momentum_3m > 3, momentum_3m < -3)},
    {'signal': 'Calm or stormy', 'reading': 'ups and downs %.0f%% a year vs %.0f%% normal' % (stormy.iloc[-1], normal),
     'call': read(stormy.iloc[-1] < normal, stormy.iloc[-1] > 1.25 * normal)},
    {'signal': 'Distance from the high', 'reading': '%.1f%% below its 1-year high' % below_high.iloc[-1],
     'call': read(below_high.iloc[-1] < 5, below_high.iloc[-1] > 10)},
    {'signal': 'Breadth (your stocks)', 'reading': '%.0f%% of your stocks above their 200-day average' % share_above,
     'call': read(share_above > 60, share_above < 40)},
])

fig, panels = subplots(2, 2, figsize=(14, 8.5))
panel = panels[0, 0]
panel.plot(spy_line.loc[signals_from:].index, spy_line.loc[signals_from:].values, color='black', label='S&P 500 (SPY)')
panel.plot(average_50.loc[signals_from:].index, average_50.loc[signals_from:].values, color='tab:orange', label='50-day average')
panel.plot(average_200.loc[signals_from:].index, average_200.loc[signals_from:].values, color='tab:purple', label='200-day average')
panel.set_title('Trend and crossover')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9)
panel = panels[0, 1]
panel.plot(stormy.loc[signals_from:].index, stormy.loc[signals_from:].values, color='tab:red')
panel.axhline(normal, color='gray', linestyle='--', label='normal (median of the past year)')
panel.axhline(1.25 * normal, color='tab:red', linestyle=':', label='stormy (25% above normal)')
panel.set_title('Calm or stormy: the last month of ups and downs (% a year)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9)
panel = panels[1, 0]
panel.fill_between(below_high.loc[signals_from:].index, 0, -below_high.loc[signals_from:].values, color='tab:red', alpha=0.4)
panel.axhline(-5, color='gray', linestyle='--', label='5% below')
panel.axhline(-10, color='tab:red', linestyle=':', label='10% below')
panel.set_title('Distance from the 1-year high (%)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9, loc='lower left')
panel = panels[1, 1]
bar_colors = []
for gap_pct in breadth.values:
    if gap_pct > 0:
        bar_colors.append('tab:green')
    else:
        bar_colors.append('tab:red')
panel.barh(breadth.index, breadth.values, color=bar_colors)
panel.axvline(0, color='black')
panel.set_title('Breadth: each of your stocks vs its own 200-day average (%)')
fig.suptitle('What should an investor do now? (through the close of %s)' % spy_line.index[-1].date(), fontsize=14, fontweight='bold')
fig.tight_layout()
show()

bulls = int((signals['call'] == '🐂 bull').sum())
bears = int((signals['call'] == '🐻 bear').sum())
if bulls - bears >= 2:
    lean = '🐂 the data leans BULL'
elif bears - bulls >= 2:
    lean = '🐻 the data leans BEAR'
else:
    lean = '➖ the data is MIXED (sideways)'
print("%d bull, %d bear, %d sideways signals: %s" % (bulls, bears, 6 - bulls - bears, lean))
signals

### Our call

Which fund do you think does best from the Nov 18 close to the **Dec 3** close? Rules of thumb (not laws): in a bull
market the 🎯 fund's extra risk tends to pay; in a bear market the calmer crowd tends to hold up better. You can
disagree with the signals as long as you can say why.

Nobody scores the call. On Dec 3 the board shows what happened, and Part 4 of your report says whether you were right.

In [ ]:
my_call = {
    'outlook': 'sideways',     # 'bull', 'bear' or 'sideways': what you think the market does next
    'pick': 'concentrated',    # which does best by Dec 3: 'concentrated', 'my fund', 'partner' or 'crowd'
    'why': '',                 # one sentence: why?
}
print("Our call:", my_call['outlook'], "| we think", my_call['pick'], "does best by Dec 3 |", my_call['why'] or '(add a reason)')

### Send the 🎯 fund to the leaderboard (one partner)

**One of you** sends it, with the other's NetID as `partner_netid`. It goes on the board as its own fund, marked 🎯,
next to your Studio 5 funds (which do not change). It counts from the next market close. The receipt prints the
fund's record link: **send it to your partner**, since you both need it for the final project.

In [ ]:
netid = ""            # the sender's UConn NetID, for example abc12345
partner_netid = ""    # your partner's NetID
theme = ""            # one line about the fund, for example "great American brands we use every week"
class_code = ""       # Dr. Dave writes it on the board

In [ ]:
SUBMIT_URL = 'https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio'
BOARD_URL = 'https://storage.googleapis.com/opim5641-leaderboard-f26/index.html'

what_if = {}
for k in range(0, len(focus_path), 5):
    what_if[str(focus_path.index[k].date())] = round(float(focus_path.iloc[k]), 2)
what_if[str(focus_path.index[-1].date())] = round(float(focus_path.iloc[-1]), 2)
spy_line_sent = {}
for k in range(0, len(spy_path), 5):
    spy_line_sent[str(spy_path.index[k].date())] = round(float(spy_path.iloc[k]), 2)
spy_line_sent[str(spy_path.index[-1].date())] = round(float(spy_path.iloc[-1]), 2)
focus_sectors = {}
for t in focus_mix:
    focus_sectors[t] = sp500_sectors.get(t, 'Other')

entry = {
    'netid': netid.strip().lower(),
    'partner_netid': partner_netid.strip().lower(),
    'nickname': concentrated['name'].strip(),
    'theme': theme.strip(),
    'reason': ("Studio 6 🎯: we think %s does best by Dec 3 (%s). %s" % (my_call['pick'], my_call['outlook'], my_call['why'].strip()))[:200],
    'class_code': class_code.strip(),
    'fund_kind': 'concentrated',
    'theses': picks,
    'portfolios': {'no_rules': focus_mix, 'with_rules': focus_mix},
    'risk_limit': concentrated['risk_limit'],
    'stock_theme': 'concentrated',
    'sectors': focus_sectors,
    'outlook': my_call['outlook'],
    'call': {'pick': my_call['pick'], 'why': my_call['why'].strip()[:200]},
    'data_window': [str(prices.index[0].date()), str(prices.index[-1].date())],
    'backtest': {'with your rules': {'ending value ($)': round(float(focus_path.iloc[-1]), 0), 'worst drop (%)': focus['worst drop (%)']},
                 'S&P 500 (SPY)': {'ending value ($)': round(float(spy_path.iloc[-1]), 0), 'worst drop (%)': spy['worst drop (%)']}},
    'backtest_paths': {'with your rules': what_if, 'S&P 500 (SPY)': spy_line_sent},
    'backtest_title': "2026 what-if: the 🎯 mix held from Jan 2 ($10,000)",
    'backtest_note': "From the Studio 6 notebook: today's 🎯 mix held through 2026 so far. A what-if, not a fair test: "
                     "the shares were picked with data that includes 2026.",
    'notebook': 'studio6',
}

missing = []
for field in ['netid', 'partner_netid', 'nickname', 'theme', 'class_code']:
    if entry[field] == '':
        missing.append(field)
for t in picks:
    if '___' in picks[t]:
        missing.append("your own thesis for " + t)
if my_call['why'].strip() == '':
    missing.append("my_call['why']")

if len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "then run this cell again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and submit.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post(SUBMIT_URL, data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        received = pd.Timestamp(receipt['received_utc']).tz_convert('America/New_York')
        print("Your 🎯 fund is in! This is what the leaderboard SAVED:")
        print("  fund:         🎯", receipt['nickname'])
        print("  sent by:      ", receipt['netid'], " (signed in as", receipt['google_email'] + ")")
        print("  received:     ", received.strftime('%a %b %d, %I:%M %p'), "Eastern")
        print("  companies:    ", receipt['holdings']['with_rules'])
        print("  it counts from the next market close (4 PM Eastern on a weekday)")
        if receipt.get('fund_page'):
            print("  its page:     ", receipt['fund_page'])
        if receipt.get('record'):
            print("  its record:   ", receipt['record'])
            print("  -> send this record link to your partner, and paste it as concentrated_record_link at the top")
        print("  the leaderboard: ", BOARD_URL)
    else:
        print("Not accepted:", receipt.get('problem'))

### Studio 6 night: before you leave

Answer in a text cell:

1. Did your fund beat the crowd in 2026 so far? Name the one bet that mattered most (sector or stock, with dollars).
2. Where is your dot on the map compared with the crowd? Was the extra risk worth it?
3. Your 🎯 fund: which company did Ms. Womack give the most, and why do you think she did?
4. Your call: which fund wins by Dec 3, and which signal convinced you?

# The final project (individual, due Friday, Dec 4, 11:59 PM)

The final project is this notebook, finished, and a 2-page report. Four steps:

1. **After the Dec 3 close**, paste all three links at the top (yours, your partner's, and the 🎯 record link),
   keep your 🎯 picks and your call the same, and **Run all**. (Do not resend the 🎯 fund.)
2. **Your report numbers.** The next cell prints every number your report needs, each with a sentence starter.
3. **Your fact sheet.** The cell after it makes `my_fund_fact_sheet.png`: page 3 of your report.
4. **The 2-page report** (a PDF on HuskyCT), in five parts. Your partner writes their own.

| Part | Length | What goes in it |
|---|---|---|
| 1 · My fund and my rules | ½ page | your stocks, your rules, and why you chose them |
| 2 · Me vs the crowd | ½ page | your rank, the risk-return map, and what drove your fund (helpers, hurters, sector bets) |
| 3 · Concentrated vs diversified | ½ page | your 🎯 fund: the theses, the what-if and live results, return vs risk, what concentration cost or earned |
| 4 · What happened | ¼ page | your call vs what happened by Dec 3, and what the signals got right or wrong |
| 5 · What I would hold on Jan 1, 2027 | ¼ page | my fund, the crowd or the 🎯 fund, and why |

Your grade does **not** depend on making money. It depends on how clearly you test and explain your choices.

**Awards (just for fun, announced after the Dec 3 close):** 🏁 **Race winner** (the most money; over a few weeks that
is mostly luck), 🛡️ **Steadiest** (beat the S&P 500 with the smallest worst drop), 🎯 **Best concentrated fund**
(the highest 🎯 fund at the Dec 3 close), and 🔬 **Best investigation** (Dr. Dave's pick from the reports).

In [ ]:
def rules_in_words(entry):
    # a Studio 5 entry's rules, in words
    if entry is None or not isinstance(entry.get('rules'), dict):
        return 'an even split, no rules'
    fund_rules = entry['rules']
    words = []
    if fund_rules.get('max_share') is not None:
        words.append('at most %.0f%% in one stock' % (100 * fund_rules['max_share']))
    if fund_rules.get('min_share') is not None:
        words.append('at least %.0f%% in any stock it holds' % (100 * fund_rules['min_share']))
    if len(fund_rules.get('must_have_sectors', [])) > 0:
        words.append('at least one stock in ' + ', '.join(fund_rules['must_have_sectors']))
    if fund_rules.get('max_stocks') is not None:
        words.append('no more than %d stocks' % fund_rules['max_stocks'])
    extra = entry.get('extra_rules') or {}
    for key in extra:
        if extra[key] not in [None, []]:
            words.append('%s: %s' % (key.replace('_', ' '), extra[key]))
    return '; '.join(words)


focus_record = None
focus_live = None
if concentrated_record_link.strip() != '':
    focus_record = read_json(concentrated_record_link)
    focus_live = live_line(focus_record)

print("REPORT NUMBERS (through the close of %s)" % prices.index[-1].date())
print()
print("Part 1 · My fund and my rules")
print("  \"My fund, %s, holds %d stocks (%s list), with these rules: %s.\"" % (my_name, len(my_mix), my_theme, rules_in_words(my_entry)))
print()
print("Part 2 · Me vs the crowd (2026 so far, each mix held from Jan 2)")
print("  \"My fund turned $10,000 into $%.0f (%+.1f%%), with a risk of %.1f%% a year.\"" % (my_path.iloc[-1], mine['2026 return (%)'], mine['risk (%)']))
print("  \"The class crowd fund turned it into $%.0f (%+.1f%%), with a risk of %.1f%%: less risky than %d of %d funds.\"" % (
    crowd_path.iloc[-1], crowd['2026 return (%)'], crowd['risk (%)'], safer_than, len(class_table)))
if hurters.iloc[0] < 0:
    weakest = "my biggest hurter was %s (-$%.0f)" % (hurters.index[0], -hurters.iloc[0])
else:
    weakest = "every stock made money, and the weakest was %s (+$%.0f)" % (hurters.index[0], hurters.iloc[0])
print("  \"My fund ranks #%d of %d on return. My biggest helper was %s (+$%.0f); %s.\"" % (
    my_rank, how_many, helpers.index[0], helpers.iloc[0], weakest))
print("  \"My biggest sector bet vs the crowd was %s (%+.0f points), which %s my fund $%.0f.\"" % (
    bets['my bet (points)'].abs().idxmax(), bets.loc[bets['my bet (points)'].abs().idxmax(), 'my bet (points)'],
    'earned' if bets.loc[bets['my bet (points)'].abs().idxmax(), 'effect on my gap ($)'] >= 0 else 'cost',
    abs(bets.loc[bets['my bet (points)'].abs().idxmax(), 'effect on my gap ($)'])))
if my_live is not None:
    print("  \"In the live race, since %s my fund went from $10,000 to $%.0f.\"" % (my_live.index[0].date(), my_live.iloc[-1]))
    if crowd_live is not None:
        print("  \"The crowd, since %s, went to $%.0f.\"" % (crowd_live.index[0].date(), crowd_live.iloc[-1]))
print()
print("Part 3 · Concentrated vs diversified")
print("  \"Our 🎯 fund holds %s (risk limit %s).\"" % (', '.join(sorted(focus_mix)), concentrated['risk_limit']))
for t in focus_mix:
    print("    %s %.0f%%: \"%s\"" % (t, 100 * focus_mix[t], picks[t]))
print("  \"In the 2026 what-if it turned $10,000 into $%.0f (%+.1f%%) with a risk of %.1f%%, vs my fund's %.1f%%.\"" % (
    focus_path.iloc[-1], focus['2026 return (%)'], focus['risk (%)'], mine['risk (%)']))
print("  \"Its worst drop was %.1f%%, vs %.1f%% for my fund and %.1f%% for the crowd.\"" % (
    focus['worst drop (%)'], mine['worst drop (%)'], crowd['worst drop (%)']))
if focus_live is not None:
    print("  \"Live, since %s, the 🎯 fund went from $10,000 to $%.0f.\"" % (focus_live.index[0].date(), focus_live.iloc[-1]))
print()
print("Part 4 · What happened")
print("  \"On Nov 18 we read the market as %s and picked %s to do best by Dec 3, because %s\"" % (
    my_call['outlook'], my_call['pick'], my_call['why'] or '___'))
print("  \"The signals said: %s.\"" % lean)
print()
print("CHECKLIST")
print("  [%s] record link pasted (live race numbers)" % ('x' if my_live is not None else ' '))
print("  [%s] 🎯 record link pasted (the 🎯 fund's live numbers)" % ('x' if focus_live is not None else ' '))
print("  [%s] our own theses (no ___ left)" % (' ' if '___' in ''.join(picks.values()) else 'x'))
print("  [%s] our call has a reason" % ('x' if my_call['why'].strip() != '' else ' '))

# 🎓 Your resume kit

Something to be proud of, made from your own numbers. The next cell makes:

* **`my_fund_fact_sheet.png`**: one page about your fund and your 🎯 fund: the risk-return map, the live race, and
  the numbers. It is page 3 of your report, and something you can show in an interview.
* **`README.md`**: a ready-made front page if you save this notebook to your own GitHub (File, Save a copy in GitHub).
* **A resume bullet** you can paste and adjust.

In Colab, the two files download automatically.

In [ ]:
fact = figure(figsize=(11, 8.5))
fact.suptitle(my_name + ' · fund fact sheet', fontsize=18, fontweight='bold', x=0.05, ha='left')
fact.text(0.05, 0.905, '%d stocks (%s) | %s | concentrated fund %s: %s' % (len(my_mix), my_theme, rules_in_words(my_entry)[:60],
                                                          concentrated['name'], ', '.join(sorted(focus_mix))),
          fontsize=9.5, color='dimgray')

map_box = fact.add_axes([0.06, 0.42, 0.52, 0.43])
draw_map(focus, map_box)
map_box.set_title('2026 so far: risk vs return (each mix held from Jan 2)', fontsize=11)
map_box.legend(fontsize=6.5, loc='lower right', framealpha=0.9)

table_box = fact.add_axes([0.63, 0.42, 0.34, 0.43])
table_box.axis('off')
rows_out = []
for label, one in [(my_name[:22], mine), ('conc.: ' + concentrated['name'][:18], focus), ('class crowd fund', crowd), ('S&P 500', spy)]:
    rows_out.append([label, '$%.0f' % one['$10,000 became'], '%.1f%%' % one['risk (%)'], '%.1f%%' % one['worst drop (%)']])
board = table_box.table(cellText=rows_out, colLabels=['fund', '2026', 'risk', 'worst drop'], loc='upper center',
                        colWidths=[0.46, 0.18, 0.16, 0.2])
board.auto_set_font_size(False)
board.set_fontsize(9)
board.scale(1, 2.2)
call_text = 'Our call (Nov 18): %s; we think %s does best by Dec 3' % (my_call['outlook'], my_call['pick'])
if my_call['why'].strip() != '':
    call_text = call_text + '\n"' + my_call['why'].strip()[:90] + '"'
table_box.text(0, 0.2, call_text, fontsize=8.5, wrap=True)

live = fact.add_axes([0.06, 0.07, 0.52, 0.27])
if my_live is not None:
    live.plot(my_live.index, my_live.values, color='tab:green', linewidth=2, label='my fund (live)')
    if focus_live is not None:
        live.plot(focus_live.index, focus_live.values, color='tab:red', linewidth=2, label='concentrated fund (live)')
    if crowd_live is not None:
        live.plot(crowd_live.index, crowd_live.values, color='tab:purple', linestyle='--', linewidth=2, label='class crowd fund')
    spy_live = 10000 * prices.loc[my_live.index[0]:, 'SPY'] / prices.loc[my_live.index[0], 'SPY']
    live.plot(spy_live.index, spy_live.values, color='black', linewidth=2, label='S&P 500')
    live.set_title('The live race on the class leaderboard', fontsize=11)
    live.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))
    live.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    live.legend(fontsize=8, loc='upper left')
else:
    live.axis('off')
    live.text(0, 0.5, 'Paste your record link at the top to add the live race.', fontsize=10)
notes = fact.add_axes([0.63, 0.07, 0.34, 0.27])
notes.axis('off')
notes.text(0, 0.95, 'Built with', fontsize=10, fontweight='bold', va='top')
notes.text(0, 0.8, 'Python, pandas, Pyomo, Bonmin, ipopt\nmean-variance optimization with business rules\n'
           'a class "crowd fund" and a risk-return map\nOPIM 5641 · UConn · Fall 2026', fontsize=9, va='top')
fact.savefig('my_fund_fact_sheet.png', dpi=150)
show()

bullet = ("Built mean-variance portfolios in Python (Pyomo, Bonmin, ipopt) with business-rule constraints; benchmarked "
          "my fund against a class 'crowd' fund on a risk-return map (2026: %+.1f%% vs %+.1f%%); designed a 3-to-6 stock "
          "concentrated fund with an investment thesis per company and ran it live against the S&P 500." % (
              mine['2026 return (%)'], crowd['2026 return (%)']))
readme = ("# " + my_name + "\n\n"
          "A portfolio I built and managed in OPIM 5641 (Business Decision Modeling) at UConn, Fall 2026.\n\n"
          "## What it does\n"
          "* Picks a portfolio of %d stocks with Ms. Womack's model: the highest average return for a risk limit, "
          "solved with Pyomo (ipopt, and Bonmin for the yes/no rules).\n"
          "* Compares it with the class crowd fund (the average of every student's fund) on a risk-return map, and "
          "explains the gap stock by stock and sector by sector.\n"
          "* A concentrated fund (🎯 %s) of %d companies, each with a one-line thesis, at most 40%% in one.\n\n"
          "## Results (2026 so far, $10,000 from Jan 2, each mix held)\n" % (len(my_mix), concentrated['name'], len(focus_mix)))
readme += "* my fund: $%.0f (risk %.1f%%)\n" % (my_path.iloc[-1], mine['risk (%)'])
readme += "* 🎯 %s: $%.0f (risk %.1f%%)\n" % (concentrated['name'], focus_path.iloc[-1], focus['risk (%)'])
readme += "* class crowd fund: $%.0f (risk %.1f%%)\n" % (crowd_path.iloc[-1], crowd['risk (%)'])
readme += "* S&P 500: $%.0f\n\n" % spy_path.iloc[-1]
readme += "![fact sheet](my_fund_fact_sheet.png)\n"
open('README.md', 'w', encoding='utf-8').write(readme)

print("RESUME BULLET (adjust it to sound like you):")
print("  " + bullet)
print()
print("saved: my_fund_fact_sheet.png and README.md")
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_fund_fact_sheet.png')
    files.download('README.md')

### 🤖 Prompts to try with Claude

Copy one into Claude (or another AI assistant) with your numbers pasted in:

* *"Here is my fund and the class crowd fund on a risk-return map. Explain to a non-finance friend why the crowd is
  less risky than most of the funds in it, in three sentences."*
* *"My sector bets vs the crowd are below. Which one bet explains most of my gap, and what would I check to see if it
  was skill or luck?"*
* *"Here are our 🎯 fund's five theses. Play devil's advocate: what is the strongest case against each company?"*
* *"Here is my resume bullet. Make it shorter and stronger without claiming anything I did not do."*